# LoopMoE 1.49B (Test Branch) — Colab TPU v5e-1 Must-Test Profile

Architecture: `V=32,768`, 2-Stream Closed-Form mHC, `128×1024 top-4 + 1 shared×1024` (Mosaic TPU Pallas MoE gather/combine), `2048×1024` 1024-aligned Engram (RowAdamW), GLM-5.4 Loop-0 KV reuse, `arXiv:2611.05724` dynamic loop (`max_loops=32`, 8-bank cyclic MLA/Router, `cap=0.5`, `ce_k=2`, `bptt_window=8`).

Only essential benchmarks (`profile_dir` shares the same single pass as `bench-steps`, zero duplicate runs):
1. **Must-Test 1 (`R=1`, 4K)**: Baseline throughput, HBM & XProf trace
2. **Must-Test 2 (`R=8` & `R=32`, 4K)**: Dynamic loop early-exit MoE (`cap=0.5`) + sampled CE (`ce_k=2`) + Truncated BPTT (`w=8`) MFU & XProf trace

In [ ]:
# 1. Clone/update Test branch and install dependencies
import os
REPO_DIR = '/content/M3_Test'
if not os.path.exists(REPO_DIR):
    !git clone -b Test https://github.com/GarD0m/M3.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin Test

!pip install -q -U -r {REPO_DIR}/requirements.txt

In [ ]:
# 2. Probe TPU in a subprocess (prevents Jupyter kernel from locking libtpu.so)
!python -c "import jax; print('Backend:', jax.default_backend(), '| Devices:', jax.devices()); [print('Memory:', d.memory_stats()) for d in jax.local_devices()]"

In [ ]:
# 3. Must-Test 1: R=1 (4K) MFU + XProf trace (single pass)
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v5e-1 \
    --seq-lens 4096 \
    --loops 1 \
    --microbatch 8 \
    --grad-accum 8 \
    --chunk-ce-size 8192 \
    --no-remat-moe \
    --no-remat-sublayers \
    --warmup-steps 1 \
    --bench-steps 3 \
    --profile-dir tpu_profiles/v5e_4k_r1

In [ ]:
# 4. Must-Test 2: R=8 & R=32 Dynamic Loop (4K) MFU + XProf trace (single pass)
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v5e-1 \
    --seq-lens 4096 \
    --loops 8 32 \
    --microbatch 4 \
    --grad-accum 8 \
    --chunk-ce-size 8192 \
    --no-remat-sublayers \
    --warmup-steps 1 \
    --bench-steps 2 \
    --profile-dir tpu_profiles/v5e_4k_dyn

In [ ]:
# 5. Package XProf traces for Perfetto (https://ui.perfetto.dev) or TensorBoard
!zip -qr /content/v5e_profile_traces.zip {REPO_DIR}/tpu_profiles
print('Saved /content/v5e_profile_traces.zip')
# from google.colab import files; files.download('/content/v5e_profile_traces.zip')
# %load_ext tensorboard
# %tensorboard --logdir {REPO_DIR}/tpu_profiles/v5e_4k_r1